Goal is to construct a prox operator for use with ADMM with a diffusion model.
The required techniques can be readily found in this paper: [A VARIATIONAL PERSPECTIVE ON SOLVING INVERSE PROBLEMS WITH DIFFUSION MODELS](https://arxiv.org/pdf/2305.04391) 

In the original paper a gradient is proposed however we will build a stochastic prox by taking 
$$ || x_0 - x_i ||^2 $$ 
as additional objective.

In [1]:
using Ferrite
using ModularEIT
using Images
using IterativeSolvers
using LinearAlgebra
using Plots
using Distributions
using Statistics
using Lux
using JLD2

In [3]:
include("model/sde.jl")

#wrap_model##0 (generic function with 1 method)

In [7]:
η = 0.15

σf = 0.0e-2 # how much noise is added to f
σg = 0.0e-2 # How much noise is added to g
# Steers prox operators:
ρ_obj = 0.0

0.0

In [8]:
itp = interpolate_array_2D(Float64.(img))
n = 63 
grid = generate_grid(Quadrilateral, (n, n));
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,2,3,∂Ω)
cond_vec = project_function_to_fem(fe, itp)
cond_vec .= min.(max.(cond_vec,1e-6),1.0)
itp = interpolate_array_2D(Float64.(img))
n = 63 
grid = generate_grid(Quadrilateral, (n, n));
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,2,3,∂Ω)
cond_vec = project_function_to_fem(fe, itp)
cond_vec .= min.(max.(cond_vec,1e-6),1.0)
G_full = real_fourier_basis(8)
rhs_dict = Dict()
Threads.@threads for i in 2:256
    M = make_boundary(G_full[:, i],64)
    itp = interpolate_array_2D(M)
    rhs_dict[i] = assemble_rhs_func(fe, itp)
end
K = assemble_L(fe, cond_vec)
K_fac = cholesky(K)
mode_dict = Dict{Int64,FerriteEITMode}()
mode_dict_no_noise = Dict{Int64,FerriteEITMode}()
@time begin
    Threads.@threads for i in 2:256
        mode_dict[i-1] = create_mode_from_g(fe, rhs_dict[i], K_fac, normalize =  true, σ_f = σf, σ_g= σg)
        mode_dict_no_noise[i-1] = create_mode_from_g(fe, rhs_dict[i], K_fac, normalize =  true)
    end
end
img_start = load("Tikhonov/1.000e-01_0.000e+00_0.000e+00_0.000e+00.png")
itp_start = interpolate_array_2D(Float64.(img_start))
σ_vec = project_function_to_fem(fe, itp_start)
σ_vec .= min.(max.(σ_vec,1e-6),1.0)
using Distributions
#σ_vec = rand(Uniform(1e-6, 1.0), fe.n)
sol = FerriteSolverState(fe, σ_vec)
prblm = FerriteProblem(fe, mode_dict, sol)
eval_points = reshape(equidistant_grid(64), :)
ph = PointEvalHandler(grid, eval_points)
f,∂f = create_f∂f(prblm, 255; gn=true)
prox_obj = create_prox_linesearch(f, ∂f, ρ_obj)

  1.703785 seconds (672.85 k allocations: 853.602 MiB, 57.00% gc time, 13.11% compilation time)


#create_prox_linesearch##0 (generic function with 2 methods)

In [9]:
function grad_A(x)
    # convert to Float 64 and shift:
    xf64 = Float64.(denormalize_image(x))
    itp64 = interpolate_array_2D(xf64)
    σ64 = project_function_to_fem(fe, itp64)
    ∂f(σ64) 
    img64 = reshape(evaluate_at_points(ph, prblm.fe.dh, ∂f(σ64)),(64,64))
    Float32.(normalize_image(img64))
end
function obj_A(x)
    # convert to Float 64 and shift:
    xf64 = Float64.(denormalize_image(x))
    itp64 = interpolate_array_2D(xf64)
    σ64 = project_function_to_fem(fe, itp64)
    f(σ64)
end

obj_A (generic function with 1 method)

In [10]:
function prox_A(x)
    # convert to Float 64 and shift:
    xf64 = Float64.(denormalize_image(x))
    itp64 = interpolate_array_2D(xf64)
    σ64 = project_function_to_fem(fe, itp64)
    x, err_x, p_x = prox(σ64)
    img64 = reshape(evaluate_at_points(ph, prblm.fe.dh, ∂f(σ64)),(64,64))
    return Float32.(normalize_image(img64)), err_x, p_x
end


prox_A (generic function with 1 method)

In [ ]:
function model_gradient(μ, model, λ, T)
    # sample t ∈ [0, T]
    t = rand(Uniform(0, T))
    αbar = ᾱ(t)

    # sample noise
    ε = randn(size(μ))

    # forward evaluate
    xt = sqrt(αbar) .* μ .+ sqrt(1 - αbar) .* ε
    ϵ_pred = model(xt, t)

    # λ_t = λ / SNR_t = λ * σ_t / α_t
    αt = sqrt(αbar)
    σt = sqrt(1 - αbar)
    λ_t = λ * σt / αt

    err = dot(ϵ_pred .- ε, μ)

    grad_diff = ϵ_pred .- ε 
    grad =  λ_t .* grad_diff

    return grad, t, err, λ
end

model_gradient (generic function with 1 method)

In [ ]:
function create_SDE_prox(model, ρ, steps,T,λ)
    prox = x₀ -> begin

            obj = 0
            xᵢ = copy(x₀)
            t_ges = 0
            for i in 1:steps
                grad, t, err, _ = model_gradient(xᵢ, model, λ, T)
                xᵢ += grad
                obj += err
                t_ges += t
            end
            obj_diff = 0.5 * ρ sum(abs2, x .- x₀)
            grad_diff = ρ .* (x .- x₀)
        return xᵢ, err, 
    end
    return prox
end